# Lexi Llama 3 8B Q5_K_M — llama.cpp server (Colab / Kaggle / any GPU host)

Serves `bartowski/Lexi-Llama-3-8B-Uncensored-GGUF` (`Lexi-Llama-3-8B-Uncensored-Q5_K_M.gguf`, ~5.73 GB)
as an **OpenAI-compatible** endpoint for ZEVORA's remote Lexi backend (`local_remote`).
ZEVORA only needs the printed `BASE_URL` + `MODEL_ID` (+ optional API key).

**Free notebook sessions are temporary — the endpoint can disappear at any time.**
No paid service is required, and no single tunnel vendor is hardcoded (cell 6).

In [ ]:
# 1. Detect GPU and show VRAM.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Install a GPU-enabled llama.cpp server release (CPU fallback if no GPU).
import shutil, subprocess
has_gpu = shutil.which('nvidia-smi') is not None
print('GPU detected:', has_gpu)
!wget -q https://github.com/ggerganov/llama.cpp/releases/download/b5902/llama-b5902-bin-ubuntu-x64.zip -O /tmp/llama.zip
!unzip -o -q /tmp/llama.zip -d /opt/llama && ls /opt/llama | head -20

In [ ]:
# 3. Download ONLY the selected GGUF file (never the whole repository).
!pip -q install huggingface_hub
from huggingface_hub import hf_hub_download
REPO = 'bartowski/Lexi-Llama-3-8B-Uncensored-GGUF'
FILENAME = 'Lexi-Llama-3-8B-Uncensored-Q5_K_M.gguf'
MODEL_PATH = hf_hub_download(repo_id=REPO, filename=FILENAME)
print('MODEL_PATH =', MODEL_PATH)

In [ ]:
# 4. Start the OpenAI-compatible llama.cpp server in the background.
# Exposes GET /v1/models and POST /v1/chat/completions on port 8080.
import os
CTX = int(os.environ.get('LEXI_CTX', '8192'))
NGPUL = os.environ.get('LEXI_N_GPU_LAYERS', '-1' if has_gpu else '0')
!nohup /opt/llama/llama-server -m "$MODEL_PATH" --host 0.0.0.0 --port 8080 --ctx-size $CTX -ngl $NGPUL > /tmp/llama-server.log 2>&1 &
!sleep 8 && tail -5 /tmp/llama-server.log

In [ ]:
# 5. Health check: /v1/models must respond and list the model.
!curl -s http://127.0.0.1:8080/v1/models | head -c 600; echo

In [ ]:
# 6. Expose a temporary public tunnel (pick ONE; cloudflared needs no account).
# Option A (recommended): cloudflared quick tunnel.
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared && chmod +x /tmp/cloudflared
!nohup /tmp/cloudflared tunnel --url http://127.0.0.1:8080 > /tmp/tunnel.log 2>&1 &
!sleep 8 && grep -o 'https://[^ ]*trycloudflare.com' /tmp/tunnel.log | head -1
# Option B: ngrok (needs free authtoken): ngrok http 8080
# Option C: localtunnel: npx localtunnel --port 8080

In [ ]:
# 7. Print the values to paste into ZEVORA (REMOTE_LOCAL_* or Providers page).
import re
log = open('/tmp/tunnel.log').read()
match = re.search(r'https://[^ ]*trycloudflare\.com', log)
public = match.group(0) if match else '<see tunnel output above>'
print('BASE_URL =', public + '/v1')
print('MODEL_ID = lexi-llama-3-8b-q4_k_m')
print('API_KEY  = (empty unless you started llama-server with --api-key)')

## Shutdown / restart

- Restart server: re-run cell 4 (kill first: `!pkill -f llama-server`).
- Stop tunnel: `!pkill -f cloudflared`.
- Full stop: Runtime → Disconnect and delete runtime.
- After any restart the public URL changes — update `REMOTE_LOCAL_BASE_URL` in ZEVORA.